# Training Faster R-CNN with k-fold cross validation

The code in here is roughly based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


## Hyperparameters


In [ ]:
from typing import Optional

# Fixed hyperparameters
BATCH_SIZE = 4
KFOLDS = 5


class ExperimentParameters:
    def __init__(
        self,
        learning_rate: float,
        use_coco_pretrained: bool,
        use_agar_pretraining: bool,
        num_epochs_agar: int,
        num_epochs_photobox: int,
        use_rotation_augmentation: bool,
        use_patch_augmentation: bool,
        max_norm: Optional[float],
    ):
        self.learning_rate = learning_rate
        self.use_coco_pretrained = use_coco_pretrained
        self.use_agar_pretraining = use_agar_pretraining
        self.num_epochs_agar = num_epochs_agar
        self.num_epochs_photobox = num_epochs_photobox
        self.use_rotation_augmentation = use_rotation_augmentation
        self.use_patch_augmentation = use_patch_augmentation
        self.max_norm = max_norm

## Trainer class


In [ ]:
from pathlib import Path
from typing import Optional

import torch
from engine import evaluate, train_one_epoch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter


def average_losses(loss_dicts: list[dict[str, torch.Tensor]]):
    loss_cls = torch.stack([loss["loss_classifier"] for loss in loss_dicts]).mean()
    loss_box_reg = torch.stack([loss["loss_box_reg"] for loss in loss_dicts]).mean()
    loss_objectness = torch.stack(
        [loss["loss_objectness"] for loss in loss_dicts]
    ).mean()
    loss_rpn_box_reg = torch.stack(
        [loss["loss_rpn_box_reg"] for loss in loss_dicts]
    ).mean()

    summed_loss = loss_cls + loss_box_reg + loss_objectness + loss_rpn_box_reg

    return {
        "classifier": loss_cls,
        "box_reg": loss_box_reg,
        "objectness": loss_objectness,
        "rpn_box_reg": loss_rpn_box_reg,
        "total": summed_loss,
    }


class Trainer:
    def __init__(
        self,
        model: torch.nn.Module,
        train_data: DataLoader,
        val_data: Optional[DataLoader],
        optimizer: torch.optim.Optimizer,
        gpu_id: int,
        writer: SummaryWriter,
        model_save_path: str,
        exp_params: ExperimentParameters,
    ) -> None:
        self.gpu_id = gpu_id
        self.model = model.to(gpu_id)
        self.train_data = train_data
        self.val_data = val_data
        self.optimizer = optimizer
        self.writer = writer

        self.should_val = val_data is not None

        self.model_save_path = model_save_path
        Path(model_save_path).parent.mkdir(parents=True, exist_ok=True)

        self.exp_params = exp_params

    def _run_epoch(self, epoch, scaler, save_hparams=False):
        # train for one epoch, printing every 10 iterations
        train_losses = train_one_epoch(
            self.model,
            self.optimizer,
            self.train_data,
            self.gpu_id,
            epoch,
            scaler=scaler,
            print_freq=10,
            max_norm=self.exp_params.max_norm,
        )

        avg_train_loss_dict = average_losses(train_losses)
        self.writer.add_scalars("train_loss", avg_train_loss_dict, epoch)

        # check loss on validation set
        if self.should_val:
            with torch.no_grad():
                val_losses = train_one_epoch(
                    self.model,
                    self.optimizer,
                    self.val_data,
                    self.gpu_id,
                    epoch,
                    print_freq=10,
                    optimize=False,
                )

                avg_val_loss_dict = average_losses(val_losses)
                self.writer.add_scalars("val_loss", avg_val_loss_dict, epoch)

                # evaluate on the validation dataset
                coco_evaluator, mae, smape = evaluate(
                    self.model, self.val_data, device=self.gpu_id
                )

                stats = coco_evaluator.coco_eval["bbox"].stats

                avg_precision_iou_50 = stats[0]
                avg_recall_iou_50 = stats[8]

                val_metrics = {
                    "avg_precision_iou_50": float(avg_precision_iou_50),
                    "avg_recall_iou_50": float(avg_recall_iou_50),
                    "mae": float(mae),
                    "smape": float(smape),
                }

                self.writer.add_scalars("val_metrics", val_metrics, epoch)

                if save_hparams:
                    self.writer.add_hparams(self.exp_params.__dict__, val_metrics)

        self.writer.flush()

    def _save_checkpoint(self, epoch):
        ckp = self.model.state_dict()
        torch.save(ckp, self.model_save_path)
        print(f"Epoch {epoch} | Training checkpoint saved at {self.model_save_path}")

    def train(self, max_epochs: int):
        scaler = torch.cuda.amp.GradScaler()

        for epoch in range(max_epochs):
            self._run_epoch(epoch, scaler, save_hparams=(epoch == max_epochs - 1))

            if epoch % 10 == 0:
                self._save_checkpoint(epoch)

        self._save_checkpoint(max_epochs)

## Data helpers


In [ ]:
import json
import utils
from torch.utils.data import Dataset
from dataset_agar import AgarBBDataset


# The box data for these IDs of the AGAR dataset have negative x values for the bounding boxes.
faulty_ids = ["5794", "11988"]


def load_id_list(path: str):
    with open(path) as f:
        list = json.load(f)
        list = [int(id) for id in list if id.isdigit() and id not in faulty_ids]

    return list


def prepare_agar_datasets():
    faulty_ids = ["5794", "11988"]

    train_list = load_id_list(
        "../AGAR_dataset/training_lists/higher_resolution_train.txt"
    )
    val_list = load_id_list("../AGAR_dataset/training_lists/higher_resolution_val.txt")

    print(f"Training on {len(train_list)} images.")
    print(f"Validating on {len(val_list)} images.")

    train_dataset = AgarBBDataset("../AGAR_dataset/dataset", train_list, is_train=True)
    val_dataset = AgarBBDataset("../AGAR_dataset/dataset", val_list, is_train=False)

    return train_dataset, val_dataset


def prepare_dataloader(dataset: Dataset, batch_size: int, is_train: bool):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=4,
        pin_memory=True,
        shuffle=is_train,
        collate_fn=utils.collate_fn,
    )

In [ ]:
import requests
import json


def send_discord_notification(message: str, data: dict):
    url = "REDACTED_DISCORD_WEBHOOK_URL"

    data = {"content": message, "username": "Notification Bot"}

    if data is not None:
        data["embeds"] = [{"description": json.dumps(data, indent=4), "title": "Data"}]

    result = requests.post(url, json=data)

## Main function


In [ ]:
import os
from datetime import datetime

from dataset_photobox import PhotoboxBBDataset
from model import get_model_instance_segmentation
from sklearn.model_selection import KFold
from torch.optim import AdamW


def main(device: str, exp_params: ExperimentParameters):
    start_time_str = datetime.now().strftime("%Y-%m-%d_%H-%M")

    # Agar Pretraining
    if exp_params.use_agar_pretraining:
        print("Training on AGAR dataset\n" + "-" * 10 + "\n")

        train_dataset, val_dataset = prepare_agar_datasets()

        train_loader = prepare_dataloader(train_dataset, BATCH_SIZE, is_train=True)
        val_dataset = prepare_dataloader(val_dataset, BATCH_SIZE, is_train=False)

        model = get_model_instance_segmentation(
            AgarBBDataset.num_classes,
            pretrained=exp_params.use_coco_pretrained,
        )

        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(params, lr=exp_params.learning_rate, weight_decay=0.01)

        writer = SummaryWriter(f"runs/{start_time_str}/agar")
        agar_model_path = f"artifacts/{start_time_str}/agar.pth"

        trainer = Trainer(
            model,
            train_loader,
            val_dataset,
            optimizer,
            device,
            writer,
            agar_model_path,
            exp_params,
        )

        trainer.train(exp_params.num_epochs_agar)

    # K-fold cross validation
    kfold = KFold(n_splits=KFOLDS, shuffle=True)

    full_train_dataset = PhotoboxBBDataset(
        "../train_data",
        is_train=True,
        rotate_augmentation=exp_params.use_rotation_augmentation,
        patch_augmentation=exp_params.use_patch_augmentation,
    )

    full_val_dataset = PhotoboxBBDataset("../train_data", is_train=False)

    for fold, (train_ids, val_ids) in enumerate(kfold.split(full_train_dataset)):
        print(f"Training fold {fold}\n" + "-" * 10 + "\n")

        train_dataset = torch.utils.data.Subset(full_train_dataset, train_ids)
        val_dataset = torch.utils.data.Subset(full_val_dataset, val_ids)

        base_model = get_model_instance_segmentation(
            PhotoboxBBDataset.num_classes,
            pretrained=(
                exp_params.use_coco_pretrained
                if not exp_params.use_agar_pretraining
                else False
            ),
        ).to(device)
        if exp_params.use_agar_pretraining:
            base_model.load_state_dict(torch.load(agar_model_path))

        train_loader = prepare_dataloader(train_dataset, BATCH_SIZE, is_train=True)
        val_loader = prepare_dataloader(val_dataset, BATCH_SIZE, is_train=False)

        params = [p for p in base_model.parameters() if p.requires_grad]
        optimizer = AdamW(params, lr=exp_params.learning_rate, weight_decay=0.01)

        writer = SummaryWriter(f"runs/{start_time_str}/fold_{fold}")

        trainer = Trainer(
            base_model,
            train_loader,
            val_loader,
            optimizer,
            device,
            writer,
            f"artifacts/{start_time_str}/fold_{fold}.pth",
            exp_params,
        )

        trainer.train(exp_params.num_epochs_photobox)

    print(f"Final training run\n" + "-" * 10 + "\n")

    model = get_model_instance_segmentation(
        PhotoboxBBDataset.num_classes,
        pretrained=(
            exp_params.use_coco_pretrained
            if not exp_params.use_agar_pretraining
            else False
        ),
    ).to(device)
    if exp_params.use_agar_pretraining:
        model.load_state_dict(torch.load(agar_model_path))

    train_loader = prepare_dataloader(full_train_dataset, BATCH_SIZE, is_train=True)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(params, lr=exp_params.learning_rate, weight_decay=0.01)

    writer = SummaryWriter(f"runs/{start_time_str}/final")

    trainer = Trainer(
        model,
        train_loader,
        None,
        optimizer,
        device,
        writer,
        f"artifacts/{start_time_str}/final.pth",
        exp_params,
    )

    trainer.train(exp_params.num_epochs_photobox)

    send_discord_notification(
        f"{os.uname().nodename} {device} finished training.",
        {
            "start_time": start_time_str,
            "params": exp_params.__dict__,
        },
    )

In [ ]:
def pos_float_env(key: str, default: float) -> float:
    val = float(os.environ.get(key, default))

    return val if val >= 0 else None


def bool_env(key: str, default: bool) -> bool:
    return bool(int(os.environ.get(key, int(default))))


def int_env(key: str, default: int) -> int:
    return int(os.environ.get(key, default))


if __name__ == "__main__":
    device = os.environ.get("CUDA_DEVICE", "cuda:0")

    exp_params = ExperimentParameters(
        learning_rate=pos_float_env("EXP_LEARNING_RATE", 0.001),
        use_coco_pretrained=bool_env("EXP_USE_COCO_PRETRAINED", True),
        use_agar_pretraining=bool_env("EXP_USE_AGAR_PRETRAINING", True),
        num_epochs_agar=int_env("EXP_NUM_EPOCHS_AGAR", 10),
        num_epochs_photobox=int_env("EXP_NUM_EPOCHS_PHOTOBOX", 100),
        use_rotation_augmentation=bool_env("EXP_USE_ROTATION_AUGMENTATION", 1),
        use_patch_augmentation=bool_env("EXP_USE_PATCH_AUGMENTATION", 0),
        max_norm=pos_float_env("EXP_MAX_NORM", 0.1),
    )

    print(exp_params.__dict__)

    main(device, exp_params)